<a href="https://colab.research.google.com/github/26b21a4270kiet-oss/Audio-noise.py/blob/main/Copy_of_Audionoisereducer_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# AUDIO NOISE REDUCER USING STFT + SVD
# B.Tech Mathematics Mini Project
# ============================================================

# Install required libraries
!pip install -q librosa soundfile matplotlib numpy scipy

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from google.colab import files
from IPython.display import Audio, display

# ------------------------------------------------------------
# 1. UPLOAD AUDIO FILE
# ------------------------------------------------------------

print("Upload your noisy audio file (.wav, .mp3, etc.)")
uploaded = files.upload()

input_file = list(uploaded.keys())[0]

print("Loaded file:", input_file)


# ------------------------------------------------------------
# 2. LOAD AUDIO
# ------------------------------------------------------------

audio, sr = librosa.load(
    input_file,
    sr=None,
    mono=True
)

print("Sampling rate:", sr, "Hz")
print("Number of samples:", len(audio))
print("Duration:", round(len(audio) / sr, 2), "seconds")


# ------------------------------------------------------------
# 3. PLAY ORIGINAL AUDIO
# ------------------------------------------------------------

print("\nOriginal / Noisy Audio:")
display(Audio(audio, rate=sr))


# ------------------------------------------------------------
# 4. COMPUTE STFT
# ------------------------------------------------------------

n_fft = 1024
hop_length = 256

stft_matrix = librosa.stft(
    audio,
    n_fft=n_fft,
    hop_length=hop_length
)

# Magnitude and phase
magnitude = np.abs(stft_matrix)
phase = np.angle(stft_matrix)

print("\nSTFT matrix shape:", magnitude.shape)


# ------------------------------------------------------------
# 5. DISPLAY ORIGINAL SPECTROGRAM
# ------------------------------------------------------------

plt.figure(figsize=(12, 5))

librosa.display.specshow(
    librosa.amplitude_to_db(magnitude, ref=np.max),
    sr=sr,
    hop_length=hop_length,
    x_axis="time",
    y_axis="hz"
)

plt.colorbar(format="%+2.0f dB")
plt.title("Original Noisy Audio - Spectrogram")
plt.xlabel("Time (seconds)")
plt.ylabel("Frequency (Hz)")
plt.show()


# ------------------------------------------------------------
# 6. APPLY SVD
# ------------------------------------------------------------

print("\nPerforming SVD...")

U, S, VT = np.linalg.svd(
    magnitude,
    full_matrices=False
)

print("SVD completed.")

print("Number of singular values:", len(S))


# ------------------------------------------------------------
# 7. DISPLAY SINGULAR VALUES
# ------------------------------------------------------------

plt.figure(figsize=(10, 5))

plt.plot(S, marker='o', markersize=3)

plt.title("Singular Values")
plt.xlabel("Singular Value Index")
plt.ylabel("Magnitude")
plt.grid(True)

plt.show()


# ------------------------------------------------------------
# 8. CHOOSE RANK
# ------------------------------------------------------------

# Number of dominant singular values to keep
k = 30

print("\nSelected rank k =", k)

if k > len(S):
    k = len(S)
    print("k was larger than available singular values.")
    print("Using k =", k)


# ------------------------------------------------------------
# 9. RANK REDUCTION
# ------------------------------------------------------------

S_reduced = np.zeros_like(S)

# Keep only first k largest singular values
S_reduced[:k] = S[:k]

print("Large singular values kept:", k)
print("Small singular values removed:", len(S) - k)


# ------------------------------------------------------------
# 10. RECONSTRUCT CLEANED MAGNITUDE
# ------------------------------------------------------------

clean_magnitude = (
    U @ np.diag(S_reduced) @ VT
)

# Prevent negative numerical values
clean_magnitude = np.maximum(
    clean_magnitude,
    0
)

print("Cleaned magnitude matrix shape:",
      clean_magnitude.shape)


# ------------------------------------------------------------
# 11. DISPLAY CLEANED SPECTROGRAM
# ------------------------------------------------------------

plt.figure(figsize=(12, 5))

librosa.display.specshow(
    librosa.amplitude_to_db(
        clean_magnitude,
        ref=np.max
    ),
    sr=sr,
    hop_length=hop_length,
    x_axis="time",
    y_axis="hz"
)

plt.colorbar(format="%+2.0f dB")
plt.title("SVD Rank-Reduced Spectrogram")
plt.xlabel("Time (seconds)")
plt.ylabel("Frequency (Hz)")

plt.show()


# ------------------------------------------------------------
# 12. RECONSTRUCT COMPLEX STFT
# ------------------------------------------------------------

clean_stft = (
    clean_magnitude *
    np.exp(1j * phase)
)


# ------------------------------------------------------------
# 13. INVERSE STFT
# ------------------------------------------------------------

clean_audio = librosa.istft(
    clean_stft,
    hop_length=hop_length
)

# Normalize audio
max_value = np.max(np.abs(clean_audio))

if max_value > 0:
    clean_audio = clean_audio / max_value


# ------------------------------------------------------------
# 14. SAVE CLEAN AUDIO
# ------------------------------------------------------------

output_file = "cleaned_audio.wav"

sf.write(
    output_file,
    clean_audio,
    sr
)

print("\nCleaned audio saved as:", output_file)


# ------------------------------------------------------------
# 15. PLAY CLEAN AUDIO
# ------------------------------------------------------------

print("\nCleaned Audio:")
display(Audio(clean_audio, rate=sr))


# ------------------------------------------------------------
# 16. DISPLAY WAVEFORMS
# ------------------------------------------------------------

time_original = np.arange(len(audio)) / sr
time_clean = np.arange(len(clean_audio)) / sr


plt.figure(figsize=(12, 5))

plt.plot(
    time_original,
    audio
)

plt.title("Original Noisy Audio Waveform")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.grid(True)

plt.show()


plt.figure(figsize=(12, 5))

plt.plot(
    time_clean,
    clean_audio
)

plt.title("Cleaned Audio Waveform")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.grid(True)

plt.show()


# ------------------------------------------------------------
# 17. DOWNLOAD CLEAN AUDIO
# ------------------------------------------------------------

files.download(output_file)

Upload your noisy audio file (.wav, .mp3, etc.)
